# Exercícios

In [1]:
import pandas as pd

df = pd.read_csv("https://dados-ml-pln.s3.sa-east-1.amazonaws.com/tweets_classificados.csv", encoding='utf-8')
df.head()

,id,data_tweet,texto,sentimento
0,0,Sun Jan 08 01:22:05 +0000 2017,���⛪ @ Catedral de Santo Antônio - Governador ...,Neutro
1,1,Sun Jan 08 01:49:01 +0000 2017,"� @ Governador Valadares, Minas Gerais https:/...",Neutro
2,2,Sun Jan 08 01:01:46 +0000 2017,"�� @ Governador Valadares, Minas Gerais https:...",Neutro
3,3,Wed Jan 04 21:43:51 +0000 2017,��� https://t.co/BnDsO34qK0,Neutro
4,4,Mon Jan 09 15:08:21 +0000 2017,��� PSOL vai questionar aumento de vereadores ...,Negativo


In [46]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)


In [9]:
df['texto'][5]

'" bom é bandido morto"\nDeputado Cabo Júlio é condenado e fica inelegível por 10 anos - Politica - Estado de Minas https://t.co/3GfAqvrFHS'

## ToDo 1

Altere as funções de tratamento de texto apresentadas em sala para que elas façam a remoção de links também. 

Crie uma nova coluna chamada texto_tratado que conterá o resultado da aplicação das funções. 

In [13]:
# resposta
import nltk
import re
import string
import unicodedata

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
    
def remove_url(text):
    url_function = r"(http.*)"
    return re.sub(re.compile(url_function), " ", text)

def normalize_accents(text):
    return unicodedata.normalize("NFKD", text).encode("ASCII", "ignore").decode("utf-8")

def normalize_str(text):
    text = text.lower()
    text = remove_url(text)
    text = remove_punctuation(text)
    text = normalize_accents(text)
    
    text = re.sub(re.compile(r" +"), " ",text)
    return " ".join([w for w in text.split()])

def remove_punctuation(text):
    punctuations = string.punctuation
    table = str.maketrans({key: " " for key in punctuations})
    text = text.translate(table)
    return text


def tokenizer(text):
    stop_words = nltk.corpus.stopwords.words("english") # portuguese, caso o dataset seja em português
    if isinstance(text, str):
        text = normalize_str(text)
        text = "".join([w for w in text if not w.isdigit()])
        text = word_tokenize(text)
        text = [x for x in text if x not in stop_words]
        text = [y for y in text if len(y) > 2]
        return " ".join([t for t in text])
    else:
        return None

    
        
    

In [12]:
df['texto'][2]

'�� @ Governador Valadares, Minas Gerais https://t.co/dPkgzVR2Qw'

In [14]:
df['texto_tradado'] = df['texto'].apply(tokenizer)

In [41]:
df

,id,data_tweet,texto,sentimento,texto_tradado
0,0,Sun Jan 08 01:22:05 +0000 2017,���⛪ @ Catedral de Santo Antônio - Governador ...,Neutro,catedral santo antonio governador valadares
1,1,Sun Jan 08 01:49:01 +0000 2017,"� @ Governador Valadares, Minas Gerais https:/...",Neutro,governador valadares minas gerais
2,2,Sun Jan 08 01:01:46 +0000 2017,"�� @ Governador Valadares, Minas Gerais https:...",Neutro,governador valadares minas gerais
4,4,Mon Jan 09 15:08:21 +0000 2017,��� PSOL vai questionar aumento de vereadores ...,Negativo,psol vai questionar aumento vereadores prefeit...
5,5,Sat Jan 07 13:47:55 +0000 2017,""" bom é bandido morto""\nDeputado Cabo Júlio é ...",Neutro,bom bandido morto deputado cabo julio condenad...
...,...,...,...,...,...
5770,8194,Thu Feb 09 11:48:07 +0000 2017,"Trio é preso suspeito de roubo, tráfico e abus...",Positivo,trio preso suspeito roubo trafico abuso sexual...
5771,8195,Thu Feb 09 12:10:19 +0000 2017,"Trio é preso suspeito de roubo, tráfico e abus...",Positivo,trio preso suspeito roubo trafico abuso sexual...
5772,8196,Thu Feb 09 12:04:17 +0000 2017,"Trio é preso suspeito de roubo, tráfico e abus...",Positivo,trio preso suspeito roubo trafico abuso sexual...
5773,8197,Thu Feb 09 12:10:04 +0000 2017,"Trio é preso suspeito de roubo, tráfico e abus...",Positivo,trio preso suspeito roubo trafico abuso sexual...


## ToDo 2

Ao fazer a remoção de links, percebemos que algumas linhas da coluna texto_tratado possuem valores faltantes. Entretanto, o Python trata eles como ''(str) e nao como Null. Assim, um simples dropna nao resolve o problema. 

Encontre uma forma de remover tais elementos.

In [19]:
#resposta
df = df[df['texto_tradado'] != ""]

In [50]:
df[df['texto_tradado'].str.contains('zmg')]['texto_tradado'].to_string()

'2230    aristotelessd invito los municipios zmg aporta...'

## ToDo 3

Separe a coluna texto_tratado em conjunto de treino e teste na proporção 70/30

In [23]:
#resposta
from sklearn.model_selection import train_test_split

X = df['texto_tradado']
y = df['sentimento']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## ToDo 4

Transforme os dados para criar a representação numérica dos textos. Use uma versão com CountVectorizer e outra com TFIDFVectorizer

In [61]:
# resposta - CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer

# Instanciando e treinando o vetorizador nos dados de treino
vect_cv = CountVectorizer(ngram_range=(1,1), lowercase=False) 
vect_cv.fit(X_train)

# Transformando os dados de texto (Treino e Teste)
CountVectorizer_train = vect_cv.transform(X_train)
CountVectorizer_test = vect_cv.transform(X_test)  # O correto é aplicar no X_test!

# Para visualizar o formato da matriz gerada
print("Formato Treino:", CountVectorizer_train.shape)
print("Formato Teste:", CountVectorizer_test.shape)

Formato Treino: (4025, 5369)
Formato Teste: (1726, 5369)


In [62]:
# resposta - TFIDFVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Instanciando e treinando o vetorizador nos dados de treino
vect_tfidf = TfidfVectorizer(ngram_range=(1,1), lowercase=False)
vect_tfidf.fit(X_train)

# Transformando os dados de texto (Treino e Teste)
TfidfVectorizer_train = vect_tfidf.transform(X_train)
TfidfVectorizer_test = vect_tfidf.transform(X_test)

# Para visualizar o formato da matriz gerada
print("Formato Treino (TF-IDF):", TfidfVectorizer_train.shape)
print("Formato Teste (TF-IDF):", TfidfVectorizer_test.shape)

Formato Treino (TF-IDF): (4025, 5369)
Formato Teste (TF-IDF): (1726, 5369)


## ToDo 5

Treine um modelo SVM nas duas abordagens e compare seus resultados

In [63]:
# resposta - CountVecorizer
import time
from sklearn import svm
from sklearn import metrics

clf = svm.SVC(kernel='linear') 
start_time = time.time()
clf.fit(CountVectorizer_train, y_train)
end_time = time.time()
print('tempo decorrido: ',end_time-start_time, 'segundos')
y_pred = clf.predict(CountVectorizer_test)


tempo decorrido:  0.506119966506958 segundos


In [ ]:
# resposta - TFIDFVectorizer

## ToDo 6
Crie uma função que lematiza as palavras da coluna texto_tratado apenas se elas forem um verbo. Depois, crie uma nova coluna chamada texto_tratado_lemma que conterá o resultado da aplicação da função na coluna texto_tratado. 

Dica: use o Corpus pt_core_news_sm como referência para determinar a classe gramatical da palavra

In [ ]:
#!pip install spacy
#!python -m spacy download pt_core_news_sm

In [ ]:
# resposta

## ToDo 7

repita os ToDo 3, ToDo 4 e ToDo 5, usando como feature a coluna texto_tratado_lemma, e veja se os resultados tiveram melhora.

In [ ]:
#resposta

In [ ]:
#resposta - CountVectorizer

In [ ]:
# resposta - TFIDFVectorizer